- Generate a data suitable for de-duplication task using AI
- Different types of duplication
- different approached to detect duplications
- create a systematic approach to handle duplicated data in a dataset

# **De-Duplication:**

**Different Types of Duplicates:**

- Exact Duplicates
- Partial Duplicates
- Near duplicates (Fuzzy Duplicates)
- Cross-Column Duplicates
- Sentiment Duplicates
- Time-Based Duplicates

In [296]:
import pandas as pd

df = pd.read_csv("/mnt/d/Pytopia/Machine Learning/Personal_ML_Notes/EDA/data/customer_dedup_dataset.csv")
df.head(10)

,customer_id,full_name,first_name,last_name,age,city,email,phone
0,1001,James Carter,James,Carter,34,London,james.carter@email.com,7700900001
1,1001,James Carter,James,Carter,34,London,james.carter@email.com,7700900001
2,1002,Emily Watson,Emily,Watson,28,Manchester,emily.watson@email.com,7700900002
3,1002,Emilly Watson,Emilly,Watson,28,Manchester,emily.watson@email.com,7700900002
4,1003,Mohammed Al-Rashid,Mohammed,Al-Rashid,45,Birmingham,m.alrashid@email.com,7700900003
5,1003,Mohammad Al Rashid,Mohammad,Al Rashid,45,Birmingham,m.alrashid@email.com,7700900003
6,1004,Sarah O'Brien,Sarah,O'Brien,31,Leeds,sarah.obrien@email.com,7700900004
7,1004,Sarah OBrien,Sarah,OBrien,31,Leeds,sarah.obrien@email.com,7700900004
8,1005,Thomas Nguyen,Thomas,Nguyen,52,Liverpool,t.nguyen@email.com,7700900005
9,1005,Tom Nguyen,Tom,Nguyen,52,Liverpool,t.nguyen@email.com,7700900005


This dataset contains information about a small e-commerce shop and we wanna preprocess this dataset and then do a de-duplication task on it.

## 1.1. Initial Preprocessing:

In [297]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  41 non-null     int64
 1   full_name    41 non-null     str  
 2   first_name   41 non-null     str  
 3   last_name    41 non-null     str  
 4   age          41 non-null     int64
 5   city         41 non-null     str  
 6   email        41 non-null     str  
 7   phone        41 non-null     int64
dtypes: int64(3), str(5)
memory usage: 2.7 KB


In [298]:
# data types
df.dtypes

customer_id    int64
full_name        str
first_name       str
last_name        str
age            int64
city             str
email            str
phone          int64
dtype: object

In [299]:
# normalizing the strings
df['full_name'] = df['full_name'].str.lower()
df['city'] = df['city'].str.lower()
df['email'] = df['email'].str.lower()
df.head(10)

,customer_id,full_name,first_name,last_name,age,city,email,phone
0,1001,james carter,James,Carter,34,london,james.carter@email.com,7700900001
1,1001,james carter,James,Carter,34,london,james.carter@email.com,7700900001
2,1002,emily watson,Emily,Watson,28,manchester,emily.watson@email.com,7700900002
3,1002,emilly watson,Emilly,Watson,28,manchester,emily.watson@email.com,7700900002
4,1003,mohammed al-rashid,Mohammed,Al-Rashid,45,birmingham,m.alrashid@email.com,7700900003
5,1003,mohammad al rashid,Mohammad,Al Rashid,45,birmingham,m.alrashid@email.com,7700900003
6,1004,sarah o'brien,Sarah,O'Brien,31,leeds,sarah.obrien@email.com,7700900004
7,1004,sarah obrien,Sarah,OBrien,31,leeds,sarah.obrien@email.com,7700900004
8,1005,thomas nguyen,Thomas,Nguyen,52,liverpool,t.nguyen@email.com,7700900005
9,1005,tom nguyen,Tom,Nguyen,52,liverpool,t.nguyen@email.com,7700900005


In [300]:
# extracting domain and username from the email is possible
df['email_username'] = df['email'].apply(lambda x: x.split("@")[0])
df['domain'] = df['email'].apply(lambda x: x.split("@")[1])
df.head(10)

,customer_id,full_name,first_name,last_name,age,city,email,phone,email_username,domain
0,1001,james carter,James,Carter,34,london,james.carter@email.com,7700900001,james.carter,email.com
1,1001,james carter,James,Carter,34,london,james.carter@email.com,7700900001,james.carter,email.com
2,1002,emily watson,Emily,Watson,28,manchester,emily.watson@email.com,7700900002,emily.watson,email.com
3,1002,emilly watson,Emilly,Watson,28,manchester,emily.watson@email.com,7700900002,emily.watson,email.com
4,1003,mohammed al-rashid,Mohammed,Al-Rashid,45,birmingham,m.alrashid@email.com,7700900003,m.alrashid,email.com
5,1003,mohammad al rashid,Mohammad,Al Rashid,45,birmingham,m.alrashid@email.com,7700900003,m.alrashid,email.com
6,1004,sarah o'brien,Sarah,O'Brien,31,leeds,sarah.obrien@email.com,7700900004,sarah.obrien,email.com
7,1004,sarah obrien,Sarah,OBrien,31,leeds,sarah.obrien@email.com,7700900004,sarah.obrien,email.com
8,1005,thomas nguyen,Thomas,Nguyen,52,liverpool,t.nguyen@email.com,7700900005,t.nguyen,email.com
9,1005,tom nguyen,Tom,Nguyen,52,liverpool,t.nguyen@email.com,7700900005,t.nguyen,email.com


In [301]:
# restructuring the data to evoid cross-column duplicates
df = df[['customer_id', 'full_name', 'age', 'city', 'email', 'phone']]
df.head(10)

,customer_id,full_name,age,city,email,phone
0,1001,james carter,34,london,james.carter@email.com,7700900001
1,1001,james carter,34,london,james.carter@email.com,7700900001
2,1002,emily watson,28,manchester,emily.watson@email.com,7700900002
3,1002,emilly watson,28,manchester,emily.watson@email.com,7700900002
4,1003,mohammed al-rashid,45,birmingham,m.alrashid@email.com,7700900003
5,1003,mohammad al rashid,45,birmingham,m.alrashid@email.com,7700900003
6,1004,sarah o'brien,31,leeds,sarah.obrien@email.com,7700900004
7,1004,sarah obrien,31,leeds,sarah.obrien@email.com,7700900004
8,1005,thomas nguyen,52,liverpool,t.nguyen@email.com,7700900005
9,1005,tom nguyen,52,liverpool,t.nguyen@email.com,7700900005


## 1.2. De-Duplication:

In [302]:
# number of the unique records in every variable
df.nunique()

customer_id    24
full_name      36
age            24
city           25
email          25
phone          24
dtype: int64

By seeing thet there is 24 unique customer ids we can say there are just `24 records` out of 50 without duplicated records.

In [303]:
# exact duplicates
df[df.duplicated(keep=False)]

,customer_id,full_name,age,city,email,phone
0,1001,james carter,34,london,james.carter@email.com,7700900001
1,1001,james carter,34,london,james.carter@email.com,7700900001
32,1019,nathan brooks,31,plymouth,n.brooks@email.com,7700900019
33,1019,nathan brooks,31,plymouth,n.brooks@email.com,7700900019


In [304]:
df = df.drop_duplicates()
print(f"shape of the dataset: {df.shape}")
df.head(10)

shape of the dataset: (39, 6)


,customer_id,full_name,age,city,email,phone
0,1001,james carter,34,london,james.carter@email.com,7700900001
2,1002,emily watson,28,manchester,emily.watson@email.com,7700900002
3,1002,emilly watson,28,manchester,emily.watson@email.com,7700900002
4,1003,mohammed al-rashid,45,birmingham,m.alrashid@email.com,7700900003
5,1003,mohammad al rashid,45,birmingham,m.alrashid@email.com,7700900003
6,1004,sarah o'brien,31,leeds,sarah.obrien@email.com,7700900004
7,1004,sarah obrien,31,leeds,sarah.obrien@email.com,7700900004
8,1005,thomas nguyen,52,liverpool,t.nguyen@email.com,7700900005
9,1005,tom nguyen,52,liverpool,t.nguyen@email.com,7700900005
10,1006,charlotte evans,23,bristol,c.evans@email.com,7700900006


In [305]:
# potential partial duplicates
def potential_partial_duplicates(subset):
	return df[df.duplicated(subset=subset, keep=False)]

potential_partial_duplicates(["customer_id", 'phone'])

,customer_id,full_name,age,city,email,phone
2,1002,emily watson,28,manchester,emily.watson@email.com,7700900002
3,1002,emilly watson,28,manchester,emily.watson@email.com,7700900002
4,1003,mohammed al-rashid,45,birmingham,m.alrashid@email.com,7700900003
5,1003,mohammad al rashid,45,birmingham,m.alrashid@email.com,7700900003
6,1004,sarah o'brien,31,leeds,sarah.obrien@email.com,7700900004
7,1004,sarah obrien,31,leeds,sarah.obrien@email.com,7700900004
8,1005,thomas nguyen,52,liverpool,t.nguyen@email.com,7700900005
9,1005,tom nguyen,52,liverpool,t.nguyen@email.com,7700900005
10,1006,charlotte evans,23,bristol,c.evans@email.com,7700900006
11,1006,charlote evans,23,bristol,c.evans@email.com,7700900006


It's better to analyze them using fuzzy duplicate techniques.

### **A Partial Duplicates Identification TasK:**

In [306]:
df = df = pd.read_csv("/mnt/d/Pytopia/Machine Learning/Personal_ML_Notes/EDA/data/customer_dedup_dataset.csv")
df.head(10)

,customer_id,full_name,first_name,last_name,age,city,email,phone
0,1001,James Carter,James,Carter,34,London,james.carter@email.com,7700900001
1,1001,James Carter,James,Carter,34,London,james.carter@email.com,7700900001
2,1002,Emily Watson,Emily,Watson,28,Manchester,emily.watson@email.com,7700900002
3,1002,Emilly Watson,Emilly,Watson,28,Manchester,emily.watson@email.com,7700900002
4,1003,Mohammed Al-Rashid,Mohammed,Al-Rashid,45,Birmingham,m.alrashid@email.com,7700900003
5,1003,Mohammad Al Rashid,Mohammad,Al Rashid,45,Birmingham,m.alrashid@email.com,7700900003
6,1004,Sarah O'Brien,Sarah,O'Brien,31,Leeds,sarah.obrien@email.com,7700900004
7,1004,Sarah OBrien,Sarah,OBrien,31,Leeds,sarah.obrien@email.com,7700900004
8,1005,Thomas Nguyen,Thomas,Nguyen,52,Liverpool,t.nguyen@email.com,7700900005
9,1005,Tom Nguyen,Tom,Nguyen,52,Liverpool,t.nguyen@email.com,7700900005


By looking at the natrue of the data we can be understand that the variable "customer_id" is very important in identifying duplicates cause we know that customer ID is a unique code for any customer. Phone number is another important feature cause phone number is a unique attribute for any customer. Using both customer_id and phone from the dataset we can indentify many partial duplicates in the dataset.

In [307]:
df = df.drop(df[df.duplicated(subset=['customer_id', 'phone'])].index)
df.head(10)

,customer_id,full_name,first_name,last_name,age,city,email,phone
0,1001,James Carter,James,Carter,34,London,james.carter@email.com,7700900001
2,1002,Emily Watson,Emily,Watson,28,Manchester,emily.watson@email.com,7700900002
4,1003,Mohammed Al-Rashid,Mohammed,Al-Rashid,45,Birmingham,m.alrashid@email.com,7700900003
6,1004,Sarah O'Brien,Sarah,O'Brien,31,Leeds,sarah.obrien@email.com,7700900004
8,1005,Thomas Nguyen,Thomas,Nguyen,52,Liverpool,t.nguyen@email.com,7700900005
10,1006,Charlotte Evans,Charlotte,Evans,23,Bristol,c.evans@email.com,7700900006
12,1007,David Singh,David,Singh,38,Nottingham,david.singh@email.com,7700900007
14,1008,Priya Patel,Priya,Patel,29,Leicester,priya.patel@email.com,7700900008
16,1009,Robert Hughes,Robert,Hughes,41,Sheffield,r.hughes@email.com,7700900009
18,1010,Aisha Bakr,Aisha,Bakr,36,Edinburgh,aisha.bakr@email.com,7700900010


Now we continue our EDA path using fuzzy detenction techniques.

In [308]:
df = pd.read_csv("/mnt/d/Pytopia/Machine Learning/Personal_ML_Notes/EDA/data/customer_dedup_dataset.csv")

# normalizing the strings
df['full_name'] = df['full_name'].str.lower()
df['city'] = df['city'].str.lower()
df['email'] = df['email'].str.lower()

# restructuring the data to evoid cross-column duplicates
df = df[['customer_id', 'full_name', 'age', 'city', 'email', 'phone']]

# removing exact duplicates
df = df.drop_duplicates().reset_index(drop=True)
print(f"shape of the dataset: {df.shape}")
df.head(10)

shape of the dataset: (39, 6)


,customer_id,full_name,age,city,email,phone
0,1001,james carter,34,london,james.carter@email.com,7700900001
1,1002,emily watson,28,manchester,emily.watson@email.com,7700900002
2,1002,emilly watson,28,manchester,emily.watson@email.com,7700900002
3,1003,mohammed al-rashid,45,birmingham,m.alrashid@email.com,7700900003
4,1003,mohammad al rashid,45,birmingham,m.alrashid@email.com,7700900003
5,1004,sarah o'brien,31,leeds,sarah.obrien@email.com,7700900004
6,1004,sarah obrien,31,leeds,sarah.obrien@email.com,7700900004
7,1005,thomas nguyen,52,liverpool,t.nguyen@email.com,7700900005
8,1005,tom nguyen,52,liverpool,t.nguyen@email.com,7700900005
9,1006,charlotte evans,23,bristol,c.evans@email.com,7700900006


### **Fuzzy Matching:**

In [309]:
from thefuzz import fuzz, process

process.extract(df['full_name'][0], df['full_name'][1:], limit=10)

[('charlote evans', 58, 10),
 ('charlotte evans', 56, 9),
 ('susan clarke', 50, 28),
 ('christopher lee', 49, 29),
 ('aisha bakr', 45, 17),
 ('aisha bakr', 45, 18),
 ('mohammed al-rashid', 45, 3),
 ('ms. laura brown', 44, 25),
 ('priya patel', 43, 13),
 ('mark taylor', 43, 27)]

In [310]:
import numpy as np
import pandas as pd

def detect_potential_fuzzy_duplicates(df, feature, scorer=fuzz.ratio, threshold=80):
	dups = {}
	for i in range(df.shape[0] - 1):

		potential_dups = np.array(process.extractBests(
			df[feature][i], df[feature][i+1:], scorer=scorer
		))
		mask = potential_dups[:, 1].astype(np.int8) > threshold
		potential_dups = potential_dups[mask]
		if len(potential_dups) != 0:
			dups[int(i)] = [int(index) for index in potential_dups[:, 2]]

	print(f"Number of the potential duplicates by {feature}: {len(dups)}\n")
	for i in dups:
		series = [df.iloc[index] for index in dups[i]]
		dataframe = pd.concat([df.iloc[i], series[0]], axis=1)
		if len(series) != 1:
			for j in range(1, len(series)):
				dataframe = pd.concat([dataframe, series[j]], axis=1)
		print(dataframe)
		print("-" * 80)


detect_potential_fuzzy_duplicates(df, feature="full_name", threshold=80)

Number of the potential duplicates by full_name: 15

                                  1                       2
customer_id                    1002                    1002
full_name              emily watson           emilly watson
age                              28                      28
city                     manchester              manchester
email        emily.watson@email.com  emily.watson@email.com
phone                    7700900002              7700900002
--------------------------------------------------------------------------------
                                3                     4
customer_id                  1003                  1003
full_name      mohammed al-rashid    mohammad al rashid
age                            45                    45
city                   birmingham            birmingham
email        m.alrashid@email.com  m.alrashid@email.com
phone                  7700900003            7700900003
------------------------------------------------------

In [311]:
from itertools import combinations
from thefuzz import fuzz

feature = "full_name"
threshold = 80

potential_duplicates = []
for (idx1, row1), (idx2, row2) in combinations(df.iterrows(), 2):
    if fuzz.ratio(row1[feature], row2[feature]) >= threshold:
        potential_duplicates.append((idx1, idx2))

potential_duplicates

[(1, 2),
 (3, 4),
 (5, 6),
 (7, 8),
 (9, 10),
 (11, 12),
 (13, 14),
 (15, 16),
 (17, 18),
 (19, 20),
 (21, 22),
 (23, 24),
 (25, 26),
 (32, 33),
 (34, 35)]

In [312]:
def fuzzy_dedup(df, feature, threshold=80, scorer=fuzz.ratio):
	potential_duplicates = []
	for (idx1, row1), (idx2, row2) in combinations(df.iterrows(), 2):
		if scorer(row1[feature], row2[feature]) >= threshold:
			potential_duplicates.append((idx1, idx2))

	print(f"Number of potential duplicates: {len(potential_duplicates)}")
	return potential_duplicates

In [313]:
fuzzy_duplicates = fuzzy_dedup(df, feature="full_name")
fuzzy_duplicates

Number of potential duplicates: 15


[(1, 2),
 (3, 4),
 (5, 6),
 (7, 8),
 (9, 10),
 (11, 12),
 (13, 14),
 (15, 16),
 (17, 18),
 (19, 20),
 (21, 22),
 (23, 24),
 (25, 26),
 (32, 33),
 (34, 35)]

In [319]:
# potential fuzzy duplicates
for idx1, idx2 in fuzzy_duplicates:
    print(df.iloc[idx1])
    print()
    print(df.iloc[idx2])
    print("-" * 50)

customer_id                      1002
full_name                emily watson
age                                28
city                       manchester
email          emily.watson@email.com
phone                      7700900002
phonetic                         E543
Name: 1, dtype: object

customer_id                      1002
full_name               emilly watson
age                                28
city                       manchester
email          emily.watson@email.com
phone                      7700900002
phonetic                         E543
Name: 2, dtype: object
--------------------------------------------------
customer_id                    1003
full_name        mohammed al-rashid
age                              45
city                     birmingham
email          m.alrashid@email.com
phone                    7700900003
phonetic                       M534
Name: 3, dtype: object

customer_id                    1003
full_name        mohammad al rashid
age                   

### **Phonetic Matching:**

In [318]:
import jellyfish

def phonetic_match(df: pd.DataFrame, feature):
	df['phonetic'] = df[feature].apply(jellyfish.soundex)
	new_df = df[df.duplicated(subset="phonetic", keep=False)].sort_values(by="phonetic")

	print(f"Number of potential duplicates: {df.duplicated(subset="phonetic").sum()}")
	return new_df

phonetic_match(df, "full_name")

Number of potential duplicates: 11


,customer_id,full_name,age,city,email,phone,phonetic
17,1010,aisha bakr,36,edinburgh,aisha.bakr@email.com,7700900010,A212
18,1010,aisha bakr,36,glasgow,aisha.bakr@email.com,7700900010,A212
9,1006,charlotte evans,23,bristol,c.evans@email.com,7700900006,C643
10,1006,charlote evans,23,bristol,c.evans@email.com,7700900006,C643
1,1002,emily watson,28,manchester,emily.watson@email.com,7700900002,E543
2,1002,emilly watson,28,manchester,emily.watson@email.com,7700900002,E543
33,1020,fatimah hassan,37,southampton,fatima.hassan@email.com,7700900020,F352
32,1020,fatima hassan,37,southampton,fatima.hassan@email.com,7700900020,F352
20,1011,george williams,55,cardiff,george.williams@email.com,7700900011,G624
19,1011,george williams,55,cardiff,gwilliams@email.com,7700900011,G624


### **Distance-Based Matching:**

In [320]:
from scipy.spatial.distance import pdist, squareform

### **Unsupervised Learning Method:**

### **Supervised Learning Method:**

### **Hybrid Method:**